# Crop-Specific Maize Monitoring — Colab (Google Earth Engine)
**Three modules:** ① Planting window · ② Risk maps · ③ Yield forecast — for the Greater Horn of Africa.

**Architecture (hybrid).** Google Earth Engine is the **spatial engine** (Sentinel-2/1 phenology,
SoilGrids, 250 m aggregation), all in the Earth Engine Python API. Each module reuses your existing `src/` code.
For separate, single-purpose notebooks see `01_planting_window` / `02_risk_monitoring` / `03_cpi_yield` / `04_flooding_waterlogging`.

### Where to start
1. **Put the `planting_pipeline` folder on your Google Drive** (so this notebook can `import src`).
2. Run the cells **top to bottom**. Section 0 authenticates GEE and mounts Drive.
3. Set your **AOI / season / year** in the config cell, then run each module.

> All cells reuse your existing `src/` code from Drive.

## Section 0 — Setup

### Stage 0 · What this notebook is

**A single notebook covering all three modules**, for a walkthrough in one sitting. The four numbered
notebooks (`01_planting_window` to `04_flooding_waterlogging`) are the same computations split up, with
more validation in each. Use those for work; use this one to show the whole chain.

**The yield ceiling is the calibrated one.** Sections 2 and 3 call `CPI.ym_for(COUNTRY, SEASON)`, which
returns the ceiling fitted to HarvestStat sub-national yields rather than the uncalibrated 6.0 t/ha
agronomic potential. Both cells print or use the same value, so they cannot drift apart. A country and
season with no calibration falls back to 6.0, or 4.5 for a short season, and the table in Module 3 says
which those are.

In [ ]:
# 0.1  Install dependencies (Colab). ~2-3 min the first time.
!pip -q install earthengine-api geemap pandas geopandas 2>/dev/null
print("installed.")

### Stage 0b · Earth Engine

**Expected output.** `EE ready: ok`. The export queue is per project; use
`indigo-proxy-484220-q8` if you intend to export.

In [ ]:
# 0.2  Authenticate + initialise Earth Engine  (use the account that owns project 'ee-manzikye')
import ee
PROJECT = "ee-manzikye"            # <-- your GEE cloud project
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()              # opens a sign-in; paste the token
    ee.Initialize(project=PROJECT)
print("EE ready:", ee.String("ok").getInfo())

### Stage 0c · Drive and the pipeline path

**Expected output.** `pipeline on path: ...` followed by the first few `src` module names. The cell
also `chdir`s into the folder, which matters: the crop calendar and coefficients are loaded by the
relative paths `config/season_calendar.csv` and `config/crop_coefficients.yaml`.

In [ ]:
# 0.3  Mount Google Drive and point at the pipeline folder (so we can import src/)
from google.colab import drive
drive.mount("/content/drive")
import sys, os
PIPE_DIR = "/content/drive/MyDrive/planting_pipeline"   # <-- adjust if you put it elsewhere
assert os.path.isdir(PIPE_DIR), f"Upload the planting_pipeline folder to Drive; not found at {PIPE_DIR}"
sys.path.insert(0, PIPE_DIR)
os.chdir(PIPE_DIR)                 # so relative config/ paths resolve
print("pipeline on path:", PIPE_DIR, "| src modules:", os.listdir("src")[:6], "...")

### Stage 0d · Configuration

**Time is in dekads.** A dekad is a third of a month, numbered 1 to 36 through the year; dekad 9 is 21
to 31 March. Seasons that cross the new year use a global dekad 1 to 72.

**`aoi_run` is the test box by default**, about 380 by 265 km over western and central Kenya. Swap in
`aoi` only after the box runs clean, and expect to move from interactive `getInfo()` calls to batch
exports when you do.

**Season windows.** Kenya long rains dekads 9 to 15, Kenya short rains 28 to 33, Ethiopia Meher 11 to
18, read from `config/season_calendar.csv`.

In [ ]:
# 0.4  Configuration — choose the country / season / year and an AOI
COUNTRY   = "Kenya"          # "Kenya" | "Ethiopia"
SEASON    = "Long rains"     # "Long rains" | "Short rains" | "Meher"
YEAR      = 2024
S1_ORBIT  = "ASCENDING"      # Sentinel-1B is gone (2022) -> ASCENDING has coverage over Kenya; try "DESCENDING" elsewhere
from run import GAUL_NAME
from src import zonal_aggregate as ZA
aoi = ZA.gaul_admin(ee, [GAUL_NAME[COUNTRY]], level=0).geometry()
# a smaller test box speeds up interactive work (central/western Kenya). Use `aoi` for the whole country.
AOI_TEST  = ee.Geometry.Rectangle([34.4, -1.2, 37.8, 1.2])
aoi_run   = AOI_TEST

# --- GEE-native map (geemap): built-in EE Layers panel (toggle + opacity), same control as the Code Editor ---
import geemap
try:
    from google.colab import output; output.enable_custom_widget_manager()   # needed for interactive geemap in Colab
except Exception:
    pass
def new_map(zoom=7):
    m = geemap.Map(add_google_map=False, basemap="SATELLITE")   # keyless Google tiles (avoids the old CARTO/xyz_to_folium bug)
    m.centerObject(aoi_run, zoom)
    return m
def ee_layer(m, image, vis, name, shown=True, opacity=1.0):
    m.addLayer(ee.Image(image), vis, name, shown, opacity)      # appears in the Layers panel
    return m
print(f"{COUNTRY} · {SEASON} · {YEAR}  ·  S1 orbit {S1_ORBIT}")

## Section 1 — Planting-window module  *(GEE)*
Cue fusion (Sentinel-2 red-edge NDRE + FPAR + SAR) → green-up SOS for the main seasons, or the
rainfall-anchored onset (CHIRPS 25/20 mm + P/PET) for the short rains, then the inception-report
**5 + 7 false-start gate** (`DRYSPELL_GATE`).

### Module 1 · Planting dekad

**Main seasons, cue fusion.** A dekadal fused greenness proxy is built from Sentinel-2 red edge, MODIS
FPAR and Sentinel-1 radar, the radar filling cloud gaps:

$$G_t=\tfrac12\big[\mathrm{unit}(\mathrm{NDRE}_t)+\mathrm{unit}(\mathrm{FPAR}_t)\big],
\qquad G_t \leftarrow \mathrm{unit}(\mathrm{RVI}_t)\ \text{where optical is missing}.$$

Start of season is the first sustained crossing of a quarter of the season amplitude, held within two
dekads of the climatological onset:

$$G_{\text{thr}}=G_{\min}+0.25(G_{\max}-G_{\min}),\qquad
\text{planting}=\mathrm{SOS}-2\ \text{dekads (maize)}.$$

**Short rains, rainfall anchored.** Green-up is too weak, so the FEWS rule is used:
$P_t\ge 25$ mm, $P_{t+1}+P_{t+2}\ge 20$ mm, and $P_t/ET_{0,t}\ge 0.5$.

**Then the 5 + 7 false-start gate**, where it is applied: at least 20 mm in the first 5 days, and no
dry spell longer than 7 days in the following 20.

**Expected values.** Kenya long rains 2024: modal planting dekad **8**, tenth to ninetieth percentile
**7 to 9**. Against farmer reports the estimate carries a bias of **−0.31 dekads** and an MAE of **1.02
dekads**, with **93 %** of counties inside two dekads. Treat one dekad as the noise floor of this
product.

In [ ]:
# 1.1  Planting dekad for the chosen season
from src import (utils, s2_preprocess as S2, s1_preprocess as S1, fusion_phenometrics as FZ,
                 ltn as LTN, planting_date as PD, wrsi_feedback as WR)
from run import crop_mask_image
kc, soil = utils.load_crop_coeffs()
rows = {(r["country"], r["season"]): r for r in utils.viable_products(utils.load_calendar("config/season_calendar.csv"))
        if r["crop"].lower() == "maize"}
r  = rows[(COUNTRY, SEASON)]
ss, se = utils.sos_window_dekads(r["sos_detection_window"])
mask = crop_mask_image(ee, COUNTRY, "maize", None)

if SEASON == "Short rains":                       # rainfall-anchored onset
    pet = WR.pet_dekadal(ee, aoi_run, YEAR); ch = WR.chirps_dekadal(ee, aoi_run, YEAR)
    planting = WR.wrsi_onset(ee, ch, ss, se, pet_ic=pet).updateMask(mask).toInt16()
else:                                             # green-up cue fusion
    s2 = S2.build_s2_dekadal(ee, aoi_run, YEAR); s1 = S1.build_s1_dekadal(ee, aoi_run, YEAR, orbit=S1_ORBIT)
    fpar = FZ.add_fpar_dekadal(ee, aoi_run, YEAR)
    g = FZ.build_fused_greenness(ee, s2, s1, fpar)
    ltn = LTN.build_ltn_prior(ee, aoi_run, ss, se)
    sos = FZ.detect_sos(ee, g, mask, ss, se, ltn_sos=ltn, ltn_pad=2)
    planting = PD.sos_to_planting(ee, sos, "maize").toInt16()

# 5+7 false-start gate (green-up seasons; short rains already carries 25/20)
if SEASON != "Short rains":
    ok = WR.dryspell_false_start(ee, aoi_run, planting, YEAR, dk_lo=ss, dk_hi=se+2)
    planting = planting.updateMask(ok)

# quick sanity: how many valid maize pixels?  (>0 => a product exists)
print("valid maize pixels:",
      planting.reduceRegion(ee.Reducer.count(), aoi_run, 250, maxPixels=int(1e13)).get("planting_dekad").getInfo())
M = new_map()
ee_layer(M, planting.clip(aoi_run), {"min": ss, "max": se+3, "palette": ["440154","3b528b","21908d","5dc863","fde725"]},
         f"Planting dekad — {SEASON}")
M   # geemap Layers panel (toggle + opacity) — no separate layer control needed

## Section 2 — Risk-maps module  *(GEE)*
GEE builds the spatial risk layers — staged **WRSI / WSI / crop-failure**, **CPI**, the
**excess / waterlogging** metrics, **fused canopy condition (FCCI)**, and **SPI-3** drought — masked to maize.

### Module 2 · Water balance, stresses and CPI

**The balance, FAO-56 and FAO-33.** Reference evaporation from ERA5-Land by Hargreaves,

$$ET_0=0.0023\,R_a\,(T_{\text{mean}}+17.8)\sqrt{T_{\max}-T_{\min}},$$

crop demand $WR_t=K_{c,t}ET_{0,t}$ with the maize curve $0.30\to1.20\to0.35$ over a 12-dekad cycle,
and a soil bucket that starts empty at planting:

$$AET_t=\min(SW_{t-1}+P_t,\ WR_t),\qquad
SW_t=\min(SW_{t-1}+P_t-AET_t,\ WHC),\qquad
\mathrm{WRSI}=100\frac{\sum AET}{\sum WR}.$$

Rainfall is CHIRPS; water-holding capacity is SoilGrids through Saxton and Rawls over a 1 m root zone.
Water above capacity is discarded, so **this index cannot see waterlogging** — that is the next cell.

**The three stresses.**

$$S_{\text{water}}=\sum_s K_{y,s}\Big(1-\tfrac{AET_s}{WR_s}\Big),\quad K_y=0.4,\,1.5,\,0.5;$$
$$S_{\text{heat}}=\min\Big(1,\,0.06\!\!\sum_{\text{flowering}}\!\!\max(T_{\max}-33,0)\Big);\qquad
S_{\text{veg}}=0.4\,(1-\mathrm{VCI}).$$

$$\mathrm{CPI}=100\,(1-S_{\text{water}})(1-S_{\text{heat}})(1-S_{\text{veg}}).$$

**Expected values.** WRSI at flowering 70 to 100 over maize in a normal Kenyan long rains; below 50 is
the FEWS crop-failure class. CPI mostly 55 to 85. **$S_{\text{heat}}$ will be zero in Kenya and that is
correct**: the 33 °C cap is a dekad-mean, and measured dekad-mean flowering maxima peak at 22.9, 25.9
and 29.3 °C across Kenya's three season regimes.

**The ceiling.** `YM = CPI.ym_for(COUNTRY, SEASON)` is fetched here and reused in Module 3, and the
cell prints it so the value behind the yield map is on the record: Kenya long rains 2.34 t/ha, short
rains 1.44, Ethiopia Meher 4.14.

In [ ]:
# 2.1  GEE risk layers — staged WRSI/WSI + CPI + excess (SPI-3 wet) for the season
from src.wrsi_waterbalance import run_wrsi_staged
from src import cpi as CPI, soil as SOIL, spi as SPI, excess as EX
whc = SOIL.get_whc(ee, aoi_run, soil, root_depth_cm=int(kc["maize"].get("root_depth_m",1.0)*100))
staged = run_wrsi_staged(ee, aoi_run, YEAR, planting, "maize", kc, soil, ss, se, whc_img=whc)
Sw = CPI.s_water(ee, staged); Sh = CPI.s_heat(ee, aoi_run, YEAR, planting, kc["maize"]["L_ini"]+kc["maize"]["L_dev"],
                                              kc["maize"]["L_ini"]+kc["maize"]["L_dev"]+kc["maize"]["L_mid"], ss, se)
Sv = CPI.s_veg(ee, aoi_run, YEAR, ss, se)
YM = CPI.ym_for(COUNTRY, SEASON)                # HarvestStat-calibrated attainable ceiling (t/ha)
cpi_img, yld = CPI.cpi(ee, Sw, Sh, Sv, ym=YM)
print(f"Ym for {COUNTRY} {SEASON}: {YM} t/ha")
M = new_map()
ee_layer(M, cpi_img.updateMask(mask).clip(aoi_run), {"min":0,"max":100,"palette":["a50026","fee08b","1a9850"]}, "CPI (GEE)")
ee_layer(M, staged["wrsi_flo"].updateMask(mask).clip(aoi_run), {"min":40,"max":100,"palette":["a50026","fee08b","1a9850"]}, "WRSI @flowering (GEE)")
M   # geemap Layers panel (toggle + opacity) — no separate layer control needed

### Module 2b · The wet side

**SPI-3 wet, validated.** $\mathbf{1}[\mathrm{SPI}_3\ge 1.5]$, McKee's *very wet* class, against the
1981 to 2020 CHIRPS climatology. A seasonal, surface anomaly.

**Aeration stress, modelled and uncalibrated.** A daily root-zone balance from SoilGrids hydrology:
water above field capacity drains at the soil's own rate, stress begins halfway from field capacity to
saturation, and the stage-weighted stress accumulates only while the soil stays wet, resetting whenever
it drains. Stage weights are **reversed** from the deficit side, because young maize is the vulnerable
stage: veg 1.00, flo 0.60, grf 0.35.

**Expected values.** The waterlogging index is **zero over most pixels in most seasons**, which is the
correct answer and not a failure. Use the ranking between places, not the number: the 4 mm per day
evapotranspiration and the 4-day scale that sets 100 are first-pass constants awaiting calibration.

In [ ]:
# 2.1b  New monitoring layers — waterlogging (AquaCrop aeration) · SPI-3 wet · fused canopy condition (FCCI)
from src import excess as EX
hy = SOIL.build_hydro_mm(ee, root_depth_cm=100)                       # FC/SAT/tau from SoilGrids+Saxton
mz = kc["maize"]; d_veg=mz["L_ini"]+mz["L_dev"]; d_flo=d_veg+mz["L_mid"]; lgp=mz["LGP_dekads"]
wl  = EX.aeration_stress_index(ee, aoi_run, planting, YEAR, d_veg, d_flo, lgp, ss, se, hy["FC_mm"], hy["SAT_mm"], hy["tau"])
wet = EX.spi3_wet(ee, aoi_run, YEAR, end_month=5)                     # SPI-3 wet anomaly (>=+1.5)
M = new_map()
ee_layer(M, wl.updateMask(mask).clip(aoi_run),  {"min":0,"max":40,"palette":["f7fbff","6baed6","08306b"]}, "Soil waterlogging — modelled")
ee_layer(M, wet.updateMask(mask).clip(aoi_run), {"min":0,"max":1,"palette":["ffffff","3690c0"]}, "SPI-3 very wet (excess)")
if SEASON != "Short rains":                                          # FCCI = peak fused greenness (green-up seasons)
    fcci = FZ.fused_condition(ee, g, mask, ss, se, lgp=lgp)
    ee_layer(M, fcci.clip(aoi_run), {"min":0,"max":100,"palette":["a50026","fee08b","1a9850"]}, "Canopy condition (fused 10-20 m)")
M   # geemap Layers panel (toggle + opacity) — no separate layer control needed

### Module 2c · SPI-3 drought

Earth Engine has no incomplete gamma function, so SPI uses the Wilson and Hilferty cube-root normal
approximation:

$$a=(\mu/\sigma)^2,\qquad \mathrm{SPI}=\Big[(P_3/\mu)^{1/3}-1+\tfrac{1}{9a}\Big]\sqrt{9a}.$$

**Classes.** −1 moderate drought, −1.5 severe, −2 extreme, and the wet mirror.

**Expected values.** SPI is roughly standard normal by construction, so about **16 % of pixels below
−1 in any year is normal**. A map where nearly everything is below −1, highlands included, is more
likely a CHIRPS gap than a drought.

**What SPI-3 is not.** It is rainfall only. It knows nothing about soil, crop stage or evaporative
demand. When SPI-3 and WRSI disagree, WRSI is the crop-relevant one; SPI-3 is the meteorological
context.

In [ ]:
# 2.2  SPI-3 meteorological drought (pure GEE, src/spi.py) — the drought layer
from src import spi as SPI
end_m = 5 if SEASON=='Long rains' else (9 if SEASON=='Meher' else 12)
spi3 = SPI.spi3(ee, aoi_run, YEAR, end_month=end_m)
M = new_map()
ee_layer(M, spi3.updateMask(mask).clip(aoi_run), {"min":-2,"max":2,"palette":["a50026","fee08b","ffffff","abd9e9","4575b4"]}, "SPI-3 (drought −/+ wet)")
M   # geemap Layers panel (toggle + opacity) — no separate layer control needed

## Section 3 — Yield-forecast module  *(GEE)*
Yield-gap framing: **yield (t/ha) = CPI/100 × Ym**, then × harvested area for production. Aggregated
to admin units. Ym is a reference potential — calibrate against observed yields (KALRO / HarvestStat).

### Module 3 · Yield

$$Y_a=\frac{\mathrm{CPI}}{100}\times Y_m,\qquad
\text{production (t)}=Y_a\times 6.25\ \text{ha per 250 m pixel}.$$

**The ceiling is fetched, not typed.** `CPI.ym_for(COUNTRY, SEASON)` returns the typical-year fit to
HarvestStat sub-national yields, so the notebook follows `src/cpi.py` whenever the calibration is
updated. The uncalibrated 6.0 t/ha it replaced over-predicted reported smallholder yields two- to
sevenfold.

| Country · season | $Y_m$ t/ha | Held-out MAE, calibrated vs uncalibrated |
|---|---|---|
| Kenya · Long rains | 2.34 | 0.68 vs 2.26 |
| Kenya · Short rains | 1.44 | 0.39 vs 1.94 |
| Ethiopia · Meher | 4.14 | 0.71 vs 1.38 |
| Rwanda · Season A | 2.61 | 0.37 vs 2.75 |
| Burundi · Season A | 1.88 | 0.71 vs 3.57 |
| Somalia · Gu | 1.02 | 0.24 vs 1.58 |

**Tanzania and South Sudan still fall back to 6.0**, because HarvestStat holds no maize yields for
either. Every country where the default could be tested shows it to be several times too high, so treat
their yield numbers as unusable in level, whatever this cell prints.

**Expected values with the calibrated ceiling.** Mean yield near **1.5 t/ha** for Kenya long rains and
near **3 t/ha** for Ethiopia Meher. The total production line is marked indicative because it assumes
every masked pixel is fully planted.

In [ ]:
# 3.1  Yield & production from the CPI computed in Section 2
from src import cpi as CPI
YM = CPI.ym_for(COUNTRY, SEASON)                # calibrated ceiling (src/cpi.py YM_CAL); falls back to
                                                # the uncalibrated 6.0 / 4.5 where no calibration exists
yield_tha = cpi_img.divide(100).multiply(YM).rename("yield_tha")
PIXEL_HA  = 6.25                                # 250 m pixel
M = new_map()
ee_layer(M, yield_tha.updateMask(mask).clip(aoi_run), {"min":0,"max":YM,"palette":["ffffcc","78c679","006837"]}, "Yield t/ha (GEE)")
display(M)   # geemap Layers panel (toggle + opacity) — no separate layer control needed

# total production over the AOI
tot = yield_tha.updateMask(mask).multiply(PIXEL_HA).reduceRegion(ee.Reducer.sum(), aoi_run, 250, maxPixels=int(1e13))
print("AOI total production (t, indicative):", tot.get("yield_tha").getInfo())
M

## Section 4 — Notes, caveats, next steps
- **Indices:** SPI-3, GDD, WRSI are computed pure-GEE (`src/spi.py`, `src/gdd_clock.py`, `src/wrsi_waterbalance.py`); this line is
  pipeline's pure-GEE versions (`src/spi.py`, `src/gdd_clock.py`). They use the same definitions but a
  different engine/resolution — expect close, not identical, values.
- **Resolution:** CHIRPS ~5.5 km and ERA5-Land ~9-11 km climate content on the 250 m grid — admin-scale,
  coarse for the 250 m grid — the GEE spatial layers stay authoritative for mapping.
- **Excess / waterlogging:** the aeration model (`src/excess.py`) is soil-water-based and *uncalibrated*;
  cross-check with the SPI-3-wet anomaly. See `WATERLOGGING_METHODOLOGY`.
- **Scale up:** swap `aoi_run = AOI_TEST` for the full-country `aoi`, and export with
  `ee.batch.Export.image.toDrive(...)` (batch) rather than interactive — see the `run_*.py` scripts.
- **Calibration:** Ym, CPI stress params, and the aeration parameters need observed-yield / crop-cut
  calibration before operational use.